# 宅建士 Phase 1: ベースライン評価 (FT前)

**目的**: ファインチューニング **前** のベースモデル `Qwen/Qwen2.5-7B-Instruct` で、宅建士2025年度50問の正答率を測定する。

**重要な前提（`docs/freeze-spec.md` §6 比較成立条件）**:
- このノートで取得するベースラインと、後続の FT後評価は **同一の eval セット・同一プロンプト・同一生成パラメータ・同一採点ロジック** を使う
- ベースモデルは **4bit (nf4) 量子化**でロード（FT後推論時も同じ量子化なので条件を揃える）
- `temperature=0.0`, `do_sample=False`, `seed=42`, `max_new_tokens=8` 固定
- 採点は「出力先頭の最初の半角数字 [1-4]」抽出方式
- few-shot / CoT は禁止（zero-shot 固定）

**実行環境**:
- Colab **GPU T4 (16GB)** を選択してください（メニュー: ランタイム > ランタイムのタイプを変更 > T4 GPU）
- 想定実行時間: GPU T4 で約 10〜20 分（モデルDL: 5〜10分、推論50問: 5〜10分）

**実行手順**:
1. ランタイムを GPU T4 に設定
2. ランタイム > すべてのセルを実行
3. 完走すると `/content/drive/MyDrive/construction-llm-ft/results/baseline_eval.json` に結果が保存される
4. ローカルでは `cd ~/construction-llm-ft && git pull` で結果を取り込める（※Drive→repoは別途 cp が必要）

**禁止事項**（CLAUDE.md 公開方針）:
- `push_to_hub` などモデル公開系の呼び出しは行わない
- `huggingface_hub.login()` は呼ばない（Qwen2.5 は anonymous DL 可能）


## 1. GPU 確認

GPU が割り当てられているか確認します。CPU 環境では Qwen2.5-7B-4bit のロードは現実的でないため、必ず GPU を有効化してください。


In [ ]:
!nvidia-smi

In [ ]:
# 念のため Python 側からも GPU を確認
try:
    import torch
    has_cuda = torch.cuda.is_available()
    print(f"torch.cuda.is_available(): {has_cuda}")
    if has_cuda:
        print(f"device count: {torch.cuda.device_count()}")
        print(f"device 0: {torch.cuda.get_device_name(0)}")
    else:
        print("[WARN] CUDA が利用できません。ランタイム > ランタイムのタイプを変更 > T4 GPU を選択してください。")
except ImportError:
    print("torch 未インストール。次のセルで依存をインストールします。")


## 2. 依存ライブラリのインストール

Qwen2.5-Instruct の推論に必要な最小限のライブラリのみインストールします。
- `transformers==4.46.*`: Qwen2.5 を安定して扱えるバージョン
- `accelerate`: `device_map="auto"` 用
- `bitsandbytes`: 4bit (nf4) 量子化

Unsloth は学習用なのでベースライン推論では不要です。


In [ ]:
!pip install -q "transformers==4.46.*" "accelerate>=0.34" "bitsandbytes>=0.44"

## 3. Google Drive のマウント

eval JSONL の読み込みと結果保存に Drive を使います。
プロジェクトフォルダは `/content/drive/MyDrive/construction-llm-ft/` を想定。

事前に Mac Mini 側から下記が同期されている必要があります（既に同期済みのはず）:
- `data/jsonl/eval/takken_eval.jsonl`


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
WORK_DIR = '/content/drive/MyDrive/construction-llm-ft'
assert os.path.isdir(WORK_DIR), f"WORK_DIR not found: {WORK_DIR}. Mac Mini側で Drive 同期されているか確認してください。"
os.chdir(WORK_DIR)
print('cwd:', os.getcwd())

# 必要ディレクトリの確認
eval_path = os.path.join(WORK_DIR, 'data/jsonl/eval/takken_eval.jsonl')
results_dir = os.path.join(WORK_DIR, 'results')
os.makedirs(results_dir, exist_ok=True)
assert os.path.isfile(eval_path), f"eval JSONL が見つかりません: {eval_path}"
print('eval path OK:', eval_path)
print('results dir:', results_dir)


## 4. 評価データの読み込み

`takken_eval.jsonl` から 50 問を読み込み、フィールド構造を確認します。


In [ ]:
import json

eval_records = []
with open(eval_path, 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        eval_records.append(json.loads(line))

print(f"loaded: {len(eval_records)} records")
assert len(eval_records) == 50, f"期待 50 件、実際 {len(eval_records)} 件"

# サンプル表示（先頭 1 件）
sample = eval_records[0]
print('keys:', list(sample.keys()))
print('id  :', sample['id'])
print('gold:', sample['output'])
print('input head:', sample['input'][:120], '...')


## 5. モデルのロード（Qwen2.5-7B-Instruct, 4bit nf4）

`docs/freeze-spec.md` §3/§6 準拠:
- `Qwen/Qwen2.5-7B-Instruct` を anonymous DL（HF_TOKEN 不要）
- 4bit nf4 量子化 + `compute_dtype=float16`
- `device_map="auto"` で T4 単一GPUへ自動配置

初回ロードはモデルDLで 5〜10 分程度かかります。


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
# pad_token を eos に揃える（generate 時の warning 回避）
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
)
model.eval()

print("model loaded.")
print("device map:", getattr(model, 'hf_device_map', 'N/A'))


## 6. 推論ヘルパー関数

`docs/freeze-spec.md` §2 のプロンプト形式（Qwen2.5 ChatML）を `tokenizer.apply_chat_template` で構築します。
system プロンプト文言は仕様書 §2 のものを固定使用。

`extract_answer` は仕様書 §4 の採点ロジック準拠（最初に出現する `[1-4]` を抽出、なければ `unparseable`）。


In [ ]:
import re

# システムプロンプト（freeze-spec.md §2 と一字一句一致させる）
SYSTEM_PROMPT = "あなたは宅地建物取引士資格試験の問題に正確に回答するアシスタントです。最も適切な選択肢の番号(1〜4)を1つだけ半角数字で答え、番号以外は出力してはいけません。"

# シード固定（temperature=0 なので決定論的だが念のため明示）
SEED = 42
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


@torch.no_grad()
def generate_answer(user_content: str) -> str:
    # Qwen2.5 chat template でプロンプトを構築し、新規生成部分のみ decode して返す
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_content},
    ]
    prompt = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=False,
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=8,
        do_sample=False,
        temperature=0.0,
        top_p=1.0,
        repetition_penalty=1.0,
        pad_token_id=tokenizer.eos_token_id,
    )
    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)


def extract_answer(raw_output: str) -> str:
    # freeze-spec.md §4 の採点ルール: 最初に出現する半角数字 [1-4] を回答とみなす
    stripped = raw_output.strip()
    m = re.search(r"[1-4]", stripped)
    return m.group(0) if m else "unparseable"


# 動作確認: 1 問だけ流す
_sample = eval_records[0]
_raw = generate_answer(_sample["input"])
_pred = extract_answer(_raw)
print(f"sample id  : {_sample['id']}")
print(f"raw output : {_raw!r}")
print(f"pred       : {_pred}")
print(f"gold       : {_sample['output']}")
print(f"correct?   : {_pred == _sample['output']}")


## 7. 評価ループ（50問）

`tqdm` で進捗表示。各問題について `generate_answer` → `extract_answer` → 正誤判定。
温度 0 の決定論的生成なので 1 回だけで十分。


In [ ]:
from tqdm.auto import tqdm

per_question = []
num_correct = 0

for rec in tqdm(eval_records, desc="evaluating"):
    qid = rec["id"]
    gold = str(rec["output"]).strip()
    user_content = rec["input"]

    raw = generate_answer(user_content)
    pred = extract_answer(raw)
    is_correct = (pred == gold)
    if is_correct:
        num_correct += 1

    per_question.append({
        "id": qid,
        "pred": pred,
        "gold": gold,
        "correct": is_correct,
        "raw": raw,
    })

num_total = len(eval_records)
accuracy = num_correct / num_total if num_total > 0 else 0.0
print(f"\nBaseline accuracy: {num_correct}/{num_total} = {accuracy:.4f}")


## 8. 結果の保存

`docs/freeze-spec.md` §7 のフォーマットに、ユーザー追加要件（`date`, `score`）を統合して保存。
保存先: `<WORK_DIR>/results/baseline_eval.json`


In [ ]:
import datetime as _dt

now = _dt.datetime.now(_dt.timezone(_dt.timedelta(hours=9)))  # JST
run_id = f"baseline_{now.strftime('%Y%m%d')}"
date_str = now.strftime("%Y-%m-%d")
timestamp = now.isoformat(timespec="seconds")

result_obj = {
    "run_id": run_id,
    "date": date_str,
    "model": MODEL_NAME,
    "adapter": None,
    "eval_set": "takken_2025",
    "num_total": num_total,
    "num_correct": num_correct,
    "accuracy": accuracy,
    "score": f"{num_correct}/{num_total}",
    "per_question": per_question,
    "config": {
        "quantization": "nf4-4bit",
        "compute_dtype": "float16",
        "temperature": 0.0,
        "top_p": 1.0,
        "do_sample": False,
        "max_new_tokens": 8,
        "repetition_penalty": 1.0,
        "seed": SEED,
    },
    "timestamp": timestamp,
}

out_path = os.path.join(results_dir, "baseline_eval.json")
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(result_obj, f, ensure_ascii=False, indent=2)

print(f"saved: {out_path}")
print(f"Baseline accuracy: {num_correct}/{num_total} = {accuracy:.4f}")


## 9. 完了後の手順

- このノートはベースラインを取るだけ。これで Phase 1 (PoC ベースライン取得) は完了
- 結果ファイル `results/baseline_eval.json` は Google Drive 側に保存される
- Mac Mini で結果を取り込みたい場合:
  ```bash
  # Mac Mini 側
  cd ~/construction-llm-ft
  # Drive→repo へ手動 cp（rclone 等の同期手段を使っている場合は不要）
  cp "/path/to/Drive/construction-llm-ft/results/baseline_eval.json" ./results/
  git add results/baseline_eval.json WORK_LOG.md
  git commit -m "Phase 1: baseline_eval result"
  git push
  ```
- 次のステップ: `notebooks/ft_qlora.ipynb` で QLoRA ファインチューニングを実行
- 比較条件は変えないこと（`docs/freeze-spec.md` §6）
